# L2 Context Compressor — 在线体验

> 把长对话压缩成结构化摘要。零安装，点「全部运行」就能用。

**怎么用：**
1. 菜单栏 → 运行时 → 全部运行
2. 在下面的文本框里粘贴你的对话历史
3. 填入 OpenAI API Key
4. 看到输出结果

In [ ]:
# @title 1. 安装依赖（只需要运行一次）
!pip install -q openai

In [ ]:
# @title 2. 粘贴对话历史 + API Key

import json
from openai import OpenAI

# --- 在这里粘贴你的对话历史 ---
CONVERSATION_JSON = """
[
  {"role": "system", "content": "你是技术助手。核心任务：..."},
  {"role": "user", "content": "..."},
  {"role": "assistant", "content": "..."}
]
"""

# --- 你的 OpenAI API Key ---
API_KEY = "sk-..."  # @param {type:"string"}

# --- 模型选择（gpt-4o-mini 足够快且便宜）---
MODEL = "gpt-4o-mini"  # @param ["gpt-4o-mini", "gpt-4o"]

conversation = json.loads(CONVERSATION_JSON)
turn_count = len([m for m in conversation if m['role'] != 'system']) // 2
input_tokens = sum(len(m.get('content', '')) for m in conversation) // 4

print(f'📂 已加载 {turn_count} 轮对话，约 {input_tokens} tokens')

In [ ]:
# @title 3. 执行压缩（点击左侧播放按钮）

client = OpenAI(api_key=API_KEY)

# 格式化对话
lines = []
turn = 0
for msg in conversation:
    role = msg.get('role', 'unknown')
    content = msg.get('content', '')
    if role == 'system':
        lines.append(f'[System]: {content}')
    else:
        turn += 1
        lines.append(f'[Turn {turn}] {role.capitalize()}: {content}')

conversation_text = '\n\n'.join(lines)

extraction_prompt = """你是对话分析员。从以下对话历史中提取关键信息。

输出严格的 JSON 格式（不要输出其他内容）：
{
  "summary": "3-5句中文摘要",
  "constraints": [{"content": "...", "established_at_turn": 轮次, "still_valid": true}],
  "key_facts": [{"content": "...", "source_turns": [轮次], "confidence": "confirmed|tentative"}],
  "decisions": [{"content": "...", "decided_at_turn": 轮次}],
  "pending_items": [{"content": "...", "raised_at_turn": 轮次, "status": "not_started|in_progress"}]
}

规则：
1. constraints: 用户设定的规则、限制。被后续推翻的标记 still_valid: false
2. key_facts: 用户提供的重要信息。confidence: confirmed=多次确认, tentative=单次提及
3. decisions: 达成的共识、做出的选择
4. pending_items: 未完成的任务
5. 闲聊、寒暄、重复内容不提取

## 对话记录
""" + conversation_text

# 截断保护
if len(extraction_prompt) > 40000:
    extraction_prompt = extraction_prompt[:40000] + '\n\n[... 对话过长，已截断 ...]'

print('📤 发送提取请求...')

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": extraction_prompt}],
    temperature=0.1,
    response_format={"type": "json_object"},
)

result = json.loads(response.choices[0].message.content)

# 格式化输出
print('\n' + '='*60)
print('📋 压缩结果')
print('='*60)
print(f'\n## 摘要\n{result.get("summary", "无")}')

constraints = [c for c in result.get('constraints', []) if c.get('still_valid', True)]
if constraints:
    print(f'\n## 活跃约束 ({len(constraints)}条)')
    for c in constraints:
        print(f'  - {c["content"]} (T{c.get("established_at_turn", "?")})')

facts = [f for f in result.get('key_facts', []) if f.get('confidence') in ('confirmed', 'tentative')]
if facts:
    print(f'\n## 关键事实 ({len(facts)}条)')
    for f in facts:
        m = '✅' if f.get('confidence') == 'confirmed' else '⚠️'
        print(f'  {m} {f["content"]}')

pending = result.get('pending_items', [])
if pending:
    print(f'\n## 待处理 ({len(pending)}条)')
    for p in pending:
        print(f'  ⬜ {p["content"]}')

print(f'\n{"="*60}')
print(f'输入: {turn_count}轮, ~{input_tokens} tokens')
print(f'输出: ~{len(json.dumps(result, ensure_ascii=False)) // 4} tokens')
print(f'压缩比: {input_tokens}:{len(json.dumps(result, ensure_ascii=False)) // 4}')

## 下一步

把上面的压缩结果粘贴到下一轮对话的 system prompt 后面，替换掉原始对话历史。

详细文档 → [v2-context/README.md](https://github.com/...)